# FFAI Async DAG Executor

This notebook demonstrates FFAI's async DAG executor, which runs a dependency
graph of prompts with **topological-parallel execution**.

New features covered:

1. **`execute_graph()`** -- run multiple prompts with automatic concurrency
2. **`AsyncFFLiteLLMClient`** -- async client for concurrent API calls
3. **Level-based parallelism** -- prompts on the same DAG level run concurrently
4. **Conditions in DAG** -- skip nodes based on prior results

This notebook uses the **real Mistral API** via `AsyncFFLiteLLMClient`.

In [ ]:
import asyncio
import sys
import time
from pathlib import Path

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / 'pyproject.toml').is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from dotenv import load_dotenv
load_dotenv()

from src.Clients.AsyncFFLiteLLMClient import AsyncFFLiteLLMClient
from src.FFAI import FFAI

async_client = AsyncFFLiteLLMClient(
    model_string="mistral/mistral-small-latest",
    temperature=0.5,
    max_tokens=512,
)

ffai = FFAI(async_client)

print(f"FFAI initialized with async client: {async_client.model}")

---

## Step 1: Simple Fan-Out (3 Independent Prompts)

Three prompts with no dependencies. They all run **concurrently** on
level 0. With a sync client this would take ~3x the time.

In [ ]:
prompts = [
    {"sequence": 0, "prompt_name": "capital_france", "prompt": "What is the capital of France? Answer in one word."},
    {"sequence": 1, "prompt_name": "capital_germany", "prompt": "What is the capital of Germany? Answer in one word."},
    {"sequence": 2, "prompt_name": "capital_spain", "prompt": "What is the capital of Spain? Answer in one word."},
]

start = time.monotonic()
result = await ffai.execute_graph(prompts)
elapsed = time.monotonic() - start

print(f"Completed in {elapsed:.1f}s")
print(f"Success: {result.success_count}, Failed: {result.failed_count}, Skipped: {result.skipped_count}")
print()
for name, r in result.results.items():
    print(f"  {name}: {r.response} [{r.status}]")

---

## Step 2: Diamond DAG (Fan-Out + Fan-In)

```
Level 0:  [research]           -- single prompt
Level 1:  [analyze] [critique]  -- two independent, run concurrently
Level 2:  [synthesize]         -- depends on both level 1 results
```

This demonstrates the core value: independent work runs in parallel,
dependent work waits for its inputs.

In [ ]:
prompts = [
    {
        "sequence": 0,
        "prompt_name": "research",
        "prompt": "List 3 key challenges in quantum computing. Be concise.",
    },
    {
        "sequence": 1,
        "prompt_name": "analyze",
        "prompt": "For each challenge, assess the current progress on a scale of 1-10.",
        "history": ["research"],
    },
    {
        "sequence": 2,
        "prompt_name": "critique",
        "prompt": "Which of these challenges is the most overhyped? Explain why.",
        "history": ["research"],
    },
    {
        "sequence": 3,
        "prompt_name": "synthesize",
        "prompt": "Synthesize the analysis and critique into a balanced one-paragraph outlook.",
        "history": ["research", "analyze", "critique"],
    },
]

graph, warnings = ffai.validate_graph(prompts)
print(f"Graph: {len(graph.nodes)} nodes, max_level={graph.max_level}")
print(f"Edges: {len(graph.edges)}")
print(f"Warnings: {warnings}")
print()

start = time.monotonic()
result = await ffai.execute_graph(prompts)
elapsed = time.monotonic() - start

print(f"Completed in {elapsed:.1f}s")
print(f"Success: {result.success_count}, Failed: {result.failed_count}")
print()
for name in ["research", "analyze", "critique", "synthesize"]:
    r = result.results.get(name)
    if r:
        preview = (r.response or "")[:120]
        print(f"--- {name} [{r.status}] ---")
        print(f"{preview}...")
        print()

---

## Step 3: Conditional Execution in the DAG

Conditions work inside the DAG just like in `generate_response()`. Nodes
with false conditions are **skipped** without calling the LLM.

In [ ]:
prompts = [
    {
        "sequence": 0,
        "prompt_name": "check_topic",
        "prompt": "Is 'artificial intelligence' a technical or philosophical topic? Answer with exactly one word.",
    },
    {
        "sequence": 1,
        "prompt_name": "technical_analysis",
        "prompt": "Explain the key ML algorithms behind modern AI. Keep it to 2 sentences.",
        "history": ["check_topic"],
        "condition": 'len({{check_topic.response}}) > 0',
    },
    {
        "sequence": 2,
        "prompt_name": "philosophical_analysis",
        "prompt": "Discuss the philosophical implications of AI consciousness.",
        "history": ["check_topic"],
        "condition": '{{check_topic.status}} == "nonexistent_status"',
    },
]

start = time.monotonic()
result = await ffai.execute_graph(prompts)
elapsed = time.monotonic() - start

print(f"Completed in {elapsed:.1f}s")
print(f"Success: {result.success_count}, Skipped: {result.skipped_count}")
print()
for name, r in result.results.items():
    preview = (r.response or "(none)")[:100]
    print(f"  {name}: [{r.status}] {preview}")

---

## Step 4: Benchmark -- Sequential vs Parallel

Compare wall-clock time for 4 independent prompts:
- **Sequential** (sync `generate_response` x4)
- **Parallel** (async `execute_graph` with level-0 concurrency)

In [ ]:
questions = [
    "Name one renewable energy source.",
    "Name one programming language.",
    "Name one planet in our solar system.",
    "Name one ocean on Earth.",
]

sync_client = AsyncFFLiteLLMClient(
    model_string="mistral/mistral-small-latest",
    temperature=0.3,
    max_tokens=64,
)

ffai_sync = FFAI(sync_client)

start = time.monotonic()
for q in questions:
    await sync_client.generate_response(prompt=q)
sequential_time = time.monotonic() - start

start = time.monotonic()
graph_prompts = [
    {"sequence": i, "prompt_name": f"q{i}", "prompt": q}
    for i, q in enumerate(questions)
]
result = await ffai.execute_graph(graph_prompts)
parallel_time = time.monotonic() - start

print(f"Sequential: {sequential_time:.1f}s")
print(f"Parallel:   {parallel_time:.1f}s")
print(f"Speedup:    {sequential_time / parallel_time:.1f}x")

---

## Step 5: Inspect the DAG Topology

Use `validate_graph()` to inspect the DAG before executing. This shows
node levels, dependency edges, and undeclared-dependency warnings.

In [ ]:
prompts = [
    {"sequence": 0, "prompt_name": "a", "prompt": "Step A"},
    {"sequence": 1, "prompt_name": "b", "prompt": "Step B", "history": ["a"]},
    {"sequence": 2, "prompt_name": "c", "prompt": "Step C", "history": ["a"]},
    {"sequence": 3, "prompt_name": "d", "prompt": "Step D", "history": ["b", "c"]},
]

graph, warnings = ffai.validate_graph(prompts)

print(f"Nodes: {len(graph.nodes)}, Max level: {graph.max_level}")
print()
for seq, node in sorted(graph.nodes.items()):
    name = node.get_prompt_name()
    deps = sorted(node.dependencies)
    print(f"  Level {node.level}: {name} (deps={deps})")

print(f"\nEdges ({len(graph.edges)}):")
for edge in graph.edges:
    from_name = graph.nodes[edge.from_seq].get_prompt_name()
    to_name = graph.nodes[edge.to_seq].get_prompt_name()
    print(f"  {from_name} -> {to_name} (source={edge.source})")

print(f"\nWarnings: {warnings}")

---

## Summary

| API | Purpose |
|-----|---------|
| `await ffai.execute_graph(prompts)` | Run a DAG of prompts with parallel execution |
| `ffai.validate_graph(prompts)` | Inspect the DAG topology without executing |
| `AsyncFFLiteLLMClient` | Async client for concurrent API calls |

| `GraphResult` field | Meaning |
|---------------------|--------|
| `.results` | Dict of prompt_name -> `ResponseResult` |
| `.success_count` | Number of successful prompts |
| `.failed_count` | Number of failed prompts |
| `.skipped_count` | Number skipped by conditions |

**Execution model:**

- Prompts are assigned to levels based on dependency depth
- All prompts on the same level run concurrently via `asyncio.gather`
- Levels execute sequentially (level N+1 waits for level N)
- Conditions are evaluated per-node using prior-level results
- Each concurrent task gets its own cloned client (no shared state)